
# Add treatment dummies to the fixed imputed quarterly panel

This notebook loads the **fixed** imputed quarterly panel from `ROOT/out`, preserves the new imputation/exclusion flag structure, creates treatment dummies using normalized municipality names, and saves the updated panel back to `out/`.

It is designed to work with the updated `quarterly_panel_imputed.*` files that include columns such as:

- `price_observed_raw`
- `contracts_observed_raw`
- `price_imputed`
- `contracts_imputed`
- `has_any_observed_price`
- `has_any_observed_contracts`
- `exclude_from_price_analysis`
- `exclude_from_contract_analysis`

The notebook keeps all existing columns and simply appends the treatment variables.


In [1]:

from pathlib import Path
import ast
import re
import unicodedata
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [2]:

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    return start

ROOT = find_project_root()
OUT_DIR = ROOT / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PKL_PATH = OUT_DIR / "quarterly_panel_imputed.pkl"
JSONL_PATH = OUT_DIR / "quarterly_panel_imputed.jsonl"

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)
print("quarterly_panel_imputed.pkl exists:", PKL_PATH.exists())
print("quarterly_panel_imputed.jsonl exists:", JSONL_PATH.exists())


ROOT: /home/venatio/documents/university/public-sector-2026
OUT_DIR: /home/venatio/documents/university/public-sector-2026/out
quarterly_panel_imputed.pkl exists: True
quarterly_panel_imputed.jsonl exists: True


In [3]:

def parse_dictish(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, dict):
        return x
    s = str(x).strip()
    if s == "":
        return np.nan
    try:
        return ast.literal_eval(s)
    except Exception:
        return np.nan

if PKL_PATH.exists():
    df = pd.read_pickle(PKL_PATH)
    source_path = PKL_PATH
elif JSONL_PATH.exists():
    df = pd.read_json(JSONL_PATH, orient="records", lines=True)
    for col in ["Population", "Migration", "Income"]:
        if col in df.columns:
            df[col] = df[col].map(parse_dictish)
    source_path = JSONL_PATH
else:
    raise FileNotFoundError(
        f"Could not find either {PKL_PATH.name} or {JSONL_PATH.name} in {OUT_DIR}"
    )

print("Loaded from:", source_path)
print("Shape:", df.shape)
display(df.head())

Loaded from: /home/venatio/documents/university/public-sector-2026/out/quarterly_panel_imputed.pkl
Shape: (56908, 17)


,Code,Municipality,Year,Quarter,Prices,Contracts,Population,Migration,Income,price_observed_raw,contracts_observed_raw,price_imputed,contracts_imputed,has_any_observed_price,has_any_observed_contracts,exclude_from_price_analysis,exclude_from_contract_analysis
0,00005,NaN,2015,Q1,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0.0,0.0,1,1
1,00005,NaN,2015,Q2,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0.0,0.0,1,1
2,00005,NaN,2015,Q3,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0.0,0.0,1,1
3,00005,NaN,2015,Q4,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0.0,0.0,1,1
4,02050,NaN,2011,Q1,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0.0,0.0,1,1


In [4]:

def normalize_text(value) -> str:
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()

    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))

    value = value.replace("’", "'").replace("`", "'")
    value = value.replace("-", " ")
    value = value.replace("/", " ")
    value = value.replace("'", "")

    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


In [5]:

treated_1_names_raw = [
    'Abrera', 'Alella', 'Amposta', 'Arenys de Mar', 'Arenys de Munt', 'Argentona',
    'Badalona', 'Badia del Vallès', 'Balaguer', 'Banyoles', 'Barberà del Vallès',
    'Barcelona', 'Berga', "la Bisbal d'Empordà", 'Blanes', 'Cabrera de Mar',
    'Cabrils', 'Caldes de Montbui', "Caldes d'Estrac", 'Calella', 'Calldetenes',
    'Cambrils', 'Canet de Mar', 'la Canonja', 'Canovelles', 'Cardedeu',
    'Castellar del Vallès', 'Castellbisbal', 'Castelldefels', 'Cerdanyola del Vallès',
    'Cervelló', 'Cervera', 'Corbera de Llobregat', 'Cornellà de Llobregat',
    'Cubelles', 'Esparreguera', 'Esplugues de Llobregat', 'Falset', 'Figueres',
    'les Franqueses del Vallès', 'la Garriga', 'Gavà', 'Girona', 'Granollers',
    'Guissona', "l'Hospitalet de Llobregat", 'Igualada', 'la Llagosta', 'Lleida',
    'Llinars del Vallès', 'Lloret de Mar', 'Malgrat de Mar', 'Manlleu', 'Manresa',
    'el Masnou', 'Matadepera', 'Mataró', 'Molins de Rei', 'Mollerussa',
    'Mollet del Vallès', 'Montcada i Reixac', 'Montgat', 'Montmeló',
    'Montornès del Vallès', 'Móra la Nova', 'Olesa de Montserrat', 'Olot',
    'Palafolls', 'Palafrugell', 'Palamós', 'Palau-solità i Plegamans', 'el Papiol',
    'Parets del Vallès', 'Pineda de Mar', 'Polinyà', 'Porqueres',
    'el Prat de Llobregat', 'Premià de Dalt', 'Premià de Mar', 'Puigcerdà', 'Reus',
    'Ripoll', 'Ripollet', 'la Roca del Vallès', 'Roquetes', 'Rubí', 'Sabadell',
    'Salou', 'Salt', 'Sant Adrià de Besòs', 'Sant Andreu de Llavaneres',
    'Sant Boi de Llobregat', 'Sant Celoni', 'Sant Climent de Llobregat',
    'Sant Cugat del Vallès', 'Sant Esteve Sesrovires', 'Sant Feliu de Guíxols',
    'Sant Feliu de Llobregat', 'Sant Fost de Campsentelles', 'Sant Fruitós de Bages',
    'Sant Joan Despí', 'Sant Just Desvern', 'Sant Pere de Ribes', 'Sant Pol de Mar',
    'Sant Quirze del Vallès', "Sant Sadurní d'Anoia", 'Sant Vicenç de Montalt',
    'Sant Vicenç dels Horts', 'Santa Coloma de Cervelló', 'Santa Coloma de Farners',
    'Santa Coloma de Gramenet', 'Santa Margarida de Montbui',
    'Santa Perpètua de Mogoda', 'Santa Susanna', 'Sarrià de Ter',
    "la Seu d'Urgell", 'Sitges', 'Solsona', 'Sort', 'Tarragona', 'Tàrrega', 'Teià',
    'Terrassa', 'Tiana', 'Tona', 'Torelló', 'Torredembarra',
    'Torrelles de Llobregat', 'Tortosa', 'Tremp', 'Vallirana', 'Valls',
    'el Vendrell', 'Vic', 'Viladecans', 'Vilafranca del Penedès',
    'Vilanova del Camí', 'Vilanova i la Geltrú', 'Vilassar de Dalt',
    'Vilassar de Mar'
]

treated_2_names_raw = [
    'Aiguafreda', 'Albinyana', 'Alcanar', 'Alcarràs', 'Alcover', "l'Aldea",
    'Almacelles', 'Alpicat', 'Altafulla', "l'Ametlla de Mar",
    "l'Ametlla del Vallès", "l'Ampolla", 'Anglès', "l'Arboç", 'Arbúcies', 'Artés',
    'Artesa de Segre', 'Bagà', 'Begues', 'Begur', 'Bellpuig', 'Bellver de Cerdanya',
    'Besalú', 'Bigues i Riells del Fai', 'la Bisbal del Penedès',
    'les Borges Blanques', 'Breda', 'Cadaqués', 'Calaf', 'Calafell',
    'Caldes de Malavella', 'Calonge i Sant Antoni', 'Canyelles',
    'Cassà de la Selva', 'Castellet i la Gornal', 'Castellgalí',
    "Castelló d'Empúries", "Castell d'Aro, Platja d'Aro i S'Agaró",
    'Castellterçol', 'Celrà', 'Centelles', 'Creixell', 'Cunit', 'Deltebre',
    'Dosrius', "l'Escala", "l'Esquirol", 'Folgueroles', 'Fornells de la Selva',
    'Gelida', 'Gironella', 'Hostalric', 'Linyola', 'Llançà', "Lliçà d'Amunt",
    'Lliçà de Vall', 'Maçanet de la Selva', 'Masquefa', 'Moià',
    'Monistrol de Montserrat', 'Montblanc', 'Montbrió del Camp',
    'Mont-roig del Camp', "Móra d'Ebre", 'el Morell', 'Navàs', 'Òdena',
    'Olèrdola', 'Olesa de Bonesvalls', 'Olivella', 'Pallejà',
    'la Palma de Cervelló', 'Pals', 'Peralada', 'Piera', 'la Pobla de Montornès',
    'el Pont de Suert', 'Ponts', 'Prats de Lluçanès', 'Puig-reig', 'la Ràpita',
    'Riells i Viabrea', 'Riudarenes', 'Riudoms', 'Roda de Berà', 'Roda de Ter',
    'Roses', 'Sant Antoni de Vilamajor', 'Sant Cebrià de Vallalta',
    'Sant Esteve de Palautordera', 'Sant Feliu de Codines',
    'Sant Hipòlit de Voltregà', 'Sant Jaume dels Domenys', "Sant Jaume d'Enveja",
    'Sant Joan de Vilatorrada', 'Sant Julià de Ramis', "Sant Llorenç d'Hortons",
    'Sant Llorenç Savall', 'Sant Martí Sarroca', 'Sant Pere de Riudebitlles',
    'Sant Pere de Torelló', 'Sant Pere de Vilamajor', 'Sant Pere Pescador',
    'Sant Quirze de Besora', 'Santa Bàrbara', 'Santa Coloma de Queralt',
    "Santa Cristina d'Aro", 'Santa Eulàlia de Ronçana',
    'Santa Maria de Palautordera', 'Santa Oliva', 'Santpedor',
    'la Selva del Camp', 'la Sénia', 'Sentmenat', 'Seva', 'Sils', 'Súria',
    'Taradell', 'Torrelles de Foix', 'Torroella de Montgrí', 'Tossa de Mar',
    'Ulldecona', 'Vacarisses', 'Vallgorguina', 'Vidreres', 'Vielha e Mijaran',
    'Viladecavalls', 'Vilafant', 'Vilanova del Vallès', "Vilobí d'Onyar",
    'Vinyols i els Arcs'
]

municipality_aliases = {
    normalize_text("el masnou"): normalize_text("Masnou, el"),
    normalize_text("el papiol"): normalize_text("Papiol, el"),
    normalize_text("el prat de llobregat"): normalize_text("Prat de Llobregat, el"),
    normalize_text("el vendrell"): normalize_text("Vendrell, el"),
    normalize_text("la bisbal demporda"): normalize_text("Bisbal d'Empordà, la"),
    normalize_text("la canonja"): normalize_text("Canonja, la"),
    normalize_text("la garriga"): normalize_text("Garriga, la"),
    normalize_text("la llagosta"): normalize_text("Llagosta, la"),
    normalize_text("la roca del valles"): normalize_text("Roca del Vallès, la"),
    normalize_text("la seu durgell"): normalize_text("Seu d'Urgell, la"),
    normalize_text("les franqueses del valles"): normalize_text("Franqueses del Vallès, les"),
    normalize_text("lhospitalet de llobregat"): normalize_text("Hospitalet de Llobregat, l'"),

    normalize_text("el morell"): normalize_text("Morell, el"),
    normalize_text("el pont de suert"): normalize_text("Pont de Suert, el"),
    normalize_text("la bisbal del penedes"): normalize_text("Bisbal del Penedès, la"),
    normalize_text("la palma de cervello"): normalize_text("Palma de Cervelló, la"),
    normalize_text("la pobla de montornes"): normalize_text("Pobla de Montornès, la"),
    normalize_text("la rapita"): normalize_text("Ràpita, la"),
    normalize_text("la selva del camp"): normalize_text("Selva del Camp, la"),
    normalize_text("la senia"): normalize_text("Sénia, la"),
    normalize_text("laldea"): normalize_text("Aldea, l'"),
    normalize_text("lametlla de mar"): normalize_text("Ametlla de Mar, l'"),
    normalize_text("lametlla del valles"): normalize_text("Ametlla del Vallès, l'"),
    normalize_text("lampolla"): normalize_text("Ampolla, l'"),
    normalize_text("larboc"): normalize_text("Arboç, l'"),
    normalize_text("les borges blanques"): normalize_text("Borges Blanques, les"),
    normalize_text("lescala"): normalize_text("Escala, l'"),
    normalize_text("lesquirol"): normalize_text("Esquirol, l'"),
}

treated_1_names = {
    municipality_aliases.get(normalize_text(x), normalize_text(x))
    for x in treated_1_names_raw
}

treated_2_names = {
    municipality_aliases.get(normalize_text(x), normalize_text(x))
    for x in treated_2_names_raw
}

print("Treated 1 municipalities:", len(treated_1_names))
print("Treated 2 municipalities:", len(treated_2_names))

overlap = treated_1_names & treated_2_names
print("Overlap across the two treatment groups:", len(overlap))
if overlap:
    print(sorted(overlap))


Treated 1 municipalities: 140
Treated 2 municipalities: 131
Overlap across the two treatment groups: 0


In [6]:

required_cols = {"Municipality", "Year", "Quarter"}
missing_required = required_cols - set(df.columns)
if missing_required:
    raise ValueError(f"Missing required columns in panel: {sorted(missing_required)}")

quarter_num_map = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
invalid_quarters = sorted(set(df["Quarter"].dropna()) - set(quarter_num_map))
if invalid_quarters:
    raise ValueError(f"Unexpected quarter labels found: {invalid_quarters}")

df = df.copy()
df.columns = [str(c).strip() for c in df.columns]
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df["municipality_norm"] = df["Municipality"].map(normalize_text)
df["quarter_num"] = df["Quarter"].map(quarter_num_map).astype("Int64")
df["time_index"] = df["Year"] * 4 + df["quarter_num"]

# Coerce the new imputation / exclusion flags if they exist
flag_cols = [
    "price_observed_raw",
    "contracts_observed_raw",
    "price_imputed",
    "contracts_imputed",
    "has_any_observed_price",
    "has_any_observed_contracts",
    "exclude_from_price_analysis",
    "exclude_from_contract_analysis",
]
for col in flag_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

treat1_start = 2024 * 4 + 2   # 2024-Q2
treat2_start = 2025 * 4 + 1   # 2025-Q1

df["Treated 1"] = (
    df["municipality_norm"].isin(treated_1_names)
    & (df["time_index"] >= treat1_start)
).astype(int)

df["Treated 2"] = (
    df["municipality_norm"].isin(treated_2_names)
    & (df["time_index"] >= treat2_start)
).astype(int)

df["Control"] = ((df["Treated 1"] == 0) & (df["Treated 2"] == 0)).astype(int)

print("Rows with Treated 1 = 1:", int(df["Treated 1"].sum()))
print("Rows with Treated 2 = 1:", int(df["Treated 2"].sum()))
print("Rows with Control = 1:", int(df["Control"].sum()))


Rows with Treated 1 = 1: 980
Rows with Treated 2 = 1: 524
Rows with Control = 1: 55404


In [7]:

panel_munis = set(df["municipality_norm"].dropna().unique())

missing_t1 = sorted(treated_1_names - panel_munis)
missing_t2 = sorted(treated_2_names - panel_munis)

print("Treated 1 municipalities not found in the panel:", len(missing_t1))
if missing_t1:
    display(pd.DataFrame({"missing_treated_1": missing_t1}).head(50))

print("Treated 2 municipalities not found in the panel:", len(missing_t2))
if missing_t2:
    display(pd.DataFrame({"missing_treated_2": missing_t2}).head(50))


Treated 1 municipalities not found in the panel: 0
Treated 2 municipalities not found in the panel: 0


In [8]:

# Diagnostics tied to the new data structure
if "exclude_from_price_analysis" in df.columns:
    muni_excl_price = (
        df.groupby(["Code", "Municipality"], as_index=False)["exclude_from_price_analysis"]
          .max()
    )
    print("Municipalities excluded from price analysis:", int(muni_excl_price["exclude_from_price_analysis"].sum()))

if "exclude_from_contract_analysis" in df.columns:
    muni_excl_contract = (
        df.groupby(["Code", "Municipality"], as_index=False)["exclude_from_contract_analysis"]
          .max()
    )
    print("Municipalities excluded from contract analysis:", int(muni_excl_contract["exclude_from_contract_analysis"].sum()))

if "has_any_observed_price" in df.columns:
    muni_has_price = (
        df.groupby(["Code", "Municipality"], as_index=False)["has_any_observed_price"]
          .max()
    )
    print("Municipalities with any observed price data:", int(muni_has_price["has_any_observed_price"].sum()))


Municipalities excluded from price analysis: 420
Municipalities excluded from contract analysis: 0
Municipalities with any observed price data: 528


In [9]:

quarter_order = pd.CategoricalDtype(categories=["Q1", "Q2", "Q3", "Q4"], ordered=True)
df["Quarter"] = df["Quarter"].astype(quarter_order)

preview_cols = [c for c in [
    "Code", "Municipality", "Year", "Quarter",
    "exclude_from_price_analysis", "exclude_from_contract_analysis",
    "Treated 1", "Treated 2", "Control"
] if c in df.columns]

display(
    df.sort_values(["Municipality", "Year", "Quarter"])[preview_cols].head(30)
)


,Code,Municipality,Year,Quarter,exclude_from_price_analysis,exclude_from_contract_analysis,Treated 1,Treated 2,Control
31948,25001,Abella de la Conca,2011,Q1,1,0,0,0,1
31949,25001,Abella de la Conca,2011,Q2,1,0,0,0,1
31950,25001,Abella de la Conca,2011,Q3,1,0,0,0,1
31951,25001,Abella de la Conca,2011,Q4,1,0,0,0,1
31952,25001,Abella de la Conca,2012,Q1,1,0,0,0,1
31953,25001,Abella de la Conca,2012,Q2,1,0,0,0,1
31954,25001,Abella de la Conca,2012,Q3,1,0,0,0,1
31955,25001,Abella de la Conca,2012,Q4,1,0,0,0,1
31956,25001,Abella de la Conca,2013,Q1,1,0,0,0,1
31957,25001,Abella de la Conca,2013,Q2,1,0,0,0,1


In [10]:

df = df.drop(columns=["municipality_norm", "quarter_num", "time_index"])
df = df.sort_values(["Code", "Year", "Quarter"]).reset_index(drop=True)

TREATED_PKL_PATH = OUT_DIR / "quarterly_panel_treated.pkl"
TREATED_JSONL_PATH = OUT_DIR / "quarterly_panel_treated.jsonl"
TREATED_CSV_PATH = OUT_DIR / "quarterly_panel_treated.csv"

df.to_pickle(TREATED_PKL_PATH)
df.to_json(TREATED_JSONL_PATH, orient="records", lines=True, force_ascii=False)
df.to_csv(TREATED_CSV_PATH, index=False)

print("Saved:")
print(" -", TREATED_PKL_PATH)
print(" -", TREATED_JSONL_PATH)
print(" -", TREATED_CSV_PATH)

display(df.head())

Saved:
 - /home/venatio/documents/university/public-sector-2026/out/quarterly_panel_treated.pkl
 - /home/venatio/documents/university/public-sector-2026/out/quarterly_panel_treated.jsonl
 - /home/venatio/documents/university/public-sector-2026/out/quarterly_panel_treated.csv


,Code,Municipality,Year,Quarter,Prices,Contracts,Population,Migration,Income,price_observed_raw,contracts_observed_raw,price_imputed,contracts_imputed,has_any_observed_price,has_any_observed_contracts,exclude_from_price_analysis,exclude_from_contract_analysis,Treated 1,Treated 2,Control
0,00005,NaN,2015,Q1,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0,0,1,1,0,0,1
1,00005,NaN,2015,Q2,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0,0,1,1,0,0,1
2,00005,NaN,2015,Q3,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0,0,1,1,0,0,1
3,00005,NaN,2015,Q4,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0,0,1,1,0,0,1
4,02050,NaN,2011,Q1,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0,0,1,1,0,0,1
